# Catcher Candidate Crop Export

Generate person candidate crops from sampled video frames using the existing YOLO pose model. The output is a folder of expanded person crops plus a CSV manifest with a blank `label` column for manual labeling.

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import random

plt.rcParams["figure.figsize"] = (12, 8)

## Configuration

Update these values before running the export. `INPUT_VIDEO_PATH_OR_FOLDER` can point to one video file or to a folder containing videos.

In [ ]:
# Input can be a single video file or a folder of videos.
INPUT_VIDEO_PATH_OR_FOLDER = "data/downloader/downloads"

# Output folder for crops, annotated frames, and the manifest.
OUTPUT_FOLDER = "data/labeling/catcher_candidates"

# Process every Nth frame through YOLO video mode.
FRAME_SAMPLE_INTERVAL = 15

# Inference resolution. Higher values preserve small player detail but run slower.
YOLO_IMAGE_SIZE = 512

# YOLO person/pose confidence threshold.
YOLO_CONFIDENCE_THRESHOLD = 0.25

# Set to an integer for quick tests, or None to process all YOLO-sampled frames.
MAX_FRAMES_TO_PROCESS = 100

# Expanded crop scale around each YOLO box. 1.5 means 50% larger than the original box.
EXPANDED_CROP_SCALE = 1.5

# Save one full-frame image per sampled frame with numbered detection boxes.
SAVE_ANNOTATED_FRAMES = False

# Existing project pose model. The existing notebooks use YOLO('yolo26n-pose.pt').
MODEL_PATH = "yolo26n-pose.pt"

In [ ]:
def resolve_path(path_like):
    path = Path(path_like).expanduser()
    if path.exists():
        return path

    notebook_relative = Path.cwd() / path
    if notebook_relative.exists():
        return notebook_relative

    repo_relative = Path.cwd().parent / path
    if repo_relative.exists():
        return repo_relative

    return path


def resolve_model_path(model_path):
    path = resolve_path(model_path)
    if path.exists():
        return path

    notebooks_path = Path.cwd() / "notebooks" / model_path
    if notebooks_path.exists():
        return notebooks_path

    return path


def resolve_output_path(path_like):
    path = Path(path_like).expanduser()
    if path.is_absolute():
        return path
    if Path.cwd().name == "notebooks":
        return Path.cwd().parent / path
    return Path.cwd() / path


VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".mkv", ".m4v"}


def get_video_paths(input_path_or_folder):
    input_path = resolve_path(input_path_or_folder)
    if input_path.is_file():
        return [input_path]
    if input_path.is_dir():
        return sorted(
            p for p in input_path.rglob("*")
            if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
        )
    raise FileNotFoundError(f"Input path does not exist: {input_path_or_folder}")


input_videos = get_video_paths(INPUT_VIDEO_PATH_OR_FOLDER)
output_dir = resolve_output_path(OUTPUT_FOLDER)
crop_dir = output_dir / "crops"
annotated_dir = output_dir / "annotated_frames"
manifest_path = output_dir / "catcher_candidate_manifest.csv"

crop_dir.mkdir(parents=True, exist_ok=True)
annotated_dir.mkdir(parents=True, exist_ok=True)

print(f"Found {len(input_videos)} video(s)")
for video_path in input_videos[:10]:
    print(video_path)
if len(input_videos) > 10:
    print(f"... plus {len(input_videos) - 10} more")
print(f"Output folder: {output_dir.resolve()}")

In [ ]:
# Load the pose model
model = YOLO(str(resolve_model_path(MODEL_PATH)))

In [ ]:
def clip_box(x1, y1, x2, y2, width, height):
    x1 = max(0, min(int(round(x1)), width - 1))
    y1 = max(0, min(int(round(y1)), height - 1))
    x2 = max(0, min(int(round(x2)), width))
    y2 = max(0, min(int(round(y2)), height))
    return x1, y1, x2, y2


def expand_box(box, scale, width, height):
    x1, y1, x2, y2 = [float(v) for v in box]
    box_w = x2 - x1
    box_h = y2 - y1
    cx = x1 + box_w / 2
    cy = y1 + box_h / 2
    new_w = box_w * scale
    new_h = box_h * scale

    return clip_box(
        cx - new_w / 2,
        cy - new_h / 2,
        cx + new_w / 2,
        cy + new_h / 2,
        width,
        height,
    )


def draw_annotated_frame(frame, boxes, confs):
    annotated = frame.copy()
    for detection_id, (box, conf) in enumerate(zip(boxes, confs)):
        x1, y1, x2, y2 = [int(round(v)) for v in box]
        cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            annotated,
            f"id={detection_id} {conf:.2f}",
            (x1, max(20, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 255, 0),
            2,
        )
    return annotated


def safe_stem(path):
    return Path(path).stem.replace(" ", "_")

In [ ]:
def result_frame_index(result, result_index, stride):
    for attr_name in ("frame", "frame_idx", "frame_id"):
        if hasattr(result, attr_name):
            value = getattr(result, attr_name)
            if value is not None:
                try:
                    return int(value)
                except (TypeError, ValueError):
                    pass

    # Ultralytics does not consistently expose source frame indices across versions.
    # With vid_stride=N, streamed results are emitted from frames 0, N, 2N, ...
    return result_index * stride


def process_video(video_path):
    video_path = Path(video_path)
    video_name = video_path.name
    video_stem = safe_stem(video_path)
    rows = []
    sampled_frames = 0
    crops_saved = 0

    try:
        results = model(
            str(video_path),
            stream=True,
            save=False,
            show=False,
            verbose=False,
            conf=YOLO_CONFIDENCE_THRESHOLD,
            imgsz=YOLO_IMAGE_SIZE,
            vid_stride=FRAME_SAMPLE_INTERVAL,
        )
    except Exception as exc:
        print(f"Skipping unreadable video: {video_path} ({exc})")
        return rows, sampled_frames, crops_saved

    try:
        for result_index, result in enumerate(results):
            if MAX_FRAMES_TO_PROCESS is not None and result_index >= MAX_FRAMES_TO_PROCESS:
                break

            sampled_frames += 1
            frame_index = result_frame_index(result, result_index, FRAME_SAMPLE_INTERVAL)
            frame = result.orig_img

            if frame is None or frame.size == 0:
                print(f"Skipping unreadable frame {frame_index} from {video_name}")
                continue

            if result.boxes is None or len(result.boxes) == 0:
                continue

            boxes = result.boxes.xyxy.cpu().numpy()
            confs = result.boxes.conf.cpu().numpy()
            height, width = frame.shape[:2]

            annotated_frame_path = ""
            if SAVE_ANNOTATED_FRAMES:
                annotated = draw_annotated_frame(frame, boxes, confs)
                annotated_frame_path = annotated_dir / f"{video_stem}_frame{frame_index:06d}_annotated.jpg"
                cv2.imwrite(str(annotated_frame_path), annotated)

            for detection_id, (box, conf) in enumerate(zip(boxes, confs)):
                x1, y1, x2, y2 = clip_box(*box, width=width, height=height)
                ex1, ey1, ex2, ey2 = expand_box(box, EXPANDED_CROP_SCALE, width, height)

                if ex2 <= ex1 or ey2 <= ey1:
                    continue

                crop = frame[ey1:ey2, ex1:ex2]
                if crop.size == 0:
                    continue

                crop_path = crop_dir / f"{video_stem}_frame{frame_index:06d}_det{detection_id:02d}.jpg"
                cv2.imwrite(str(crop_path), crop)
                crops_saved += 1

                rows.append({
                    "video_name": video_name,
                    "video_path": str(video_path),
                    "frame_index": frame_index,
                    "detection_id": detection_id,
                    "crop_path": str(crop_path),
                    "annotated_frame_path": str(annotated_frame_path),
                    "bbox_x1": x1,
                    "bbox_y1": y1,
                    "bbox_x2": x2,
                    "bbox_y2": y2,
                    "yolo_confidence": float(conf),
                    "label": "",
                })
    except Exception as exc:
        print(f"Stopped early while processing {video_path}: {exc}")

    return rows, sampled_frames, crops_saved

In [ ]:
all_rows = []
total_frames_sampled = 0
total_crops_saved = 0

for video_path in input_videos:
    print(f"Processing {video_path}")
    rows, sampled_frames, crops_saved = process_video(video_path)
    all_rows.extend(rows)
    total_frames_sampled += sampled_frames
    total_crops_saved += crops_saved
    print(f"  sampled frames: {sampled_frames} | crops saved: {crops_saved}")

manifest_df = pd.DataFrame(all_rows, columns=[
    "video_name",
    "video_path",
    "frame_index",
    "detection_id",
    "crop_path",
    "annotated_frame_path",
    "bbox_x1",
    "bbox_y1",
    "bbox_x2",
    "bbox_y2",
    "yolo_confidence",
    "label",
])
manifest_df.to_csv(manifest_path, index=False)

print("\nSummary")
print(f"Videos processed: {len(input_videos)}")
print(f"Frames sampled: {total_frames_sampled}")
print(f"Crops saved: {total_crops_saved}")
print(f"CSV manifest: {manifest_path.resolve()}")

## Inspect Sample Outputs

In [ ]:
if manifest_df.empty:
    print("No crops were saved. Try lowering YOLO_CONFIDENCE_THRESHOLD, using a different video, or sampling more frames.")
else:
    sample_rows = manifest_df.sample(min(6, len(manifest_df)), random_state=42)
    sample_rows

In [ ]:
if not manifest_df.empty:
    annotated_samples = [p for p in sample_rows["annotated_frame_path"].dropna().unique() if p]
    if annotated_samples:
        annotated_path = annotated_samples[0]
        annotated = cv2.imread(annotated_path)
        if annotated is not None:
            plt.figure(figsize=(12, 8))
            plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
            plt.title(Path(annotated_path).name)
            plt.axis("off")
            plt.show()
        else:
            print(f"Could not read annotated frame: {annotated_path}")

In [ ]:
if not manifest_df.empty:
    crop_paths = sample_rows["crop_path"].tolist()
    cols = min(3, len(crop_paths))
    rows = (len(crop_paths) + cols - 1) // cols
    plt.figure(figsize=(4 * cols, 4 * rows))

    for i, crop_path in enumerate(crop_paths, start=1):
        crop = cv2.imread(crop_path)
        plt.subplot(rows, cols, i)
        if crop is None:
            plt.text(0.5, 0.5, "Unreadable crop", ha="center", va="center")
        else:
            plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        plt.title(Path(crop_path).name, fontsize=9)
        plt.axis("off")

    plt.tight_layout()
    plt.show()